In [1]:
#!pip uninstall -y keras tensorflow tensorflow-model-optimization
#!pip install tensorflow==2.12 tensorflow-model-optimization

In [2]:
# ============================
# Set backend early
# ============================
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

# ============================
# Import libraries
# ============================
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import Input, Model

import tensorflow_model_optimization as tfmot



---


# Problem 1: DNN and Wine Classification (60 points)

In [4]:
# Load dataset
column_names = [
    'Class', 'Alcohol', 'Malic acid', 'Ash', 'Alcalinity of ash', 'Magnesium',
    'Total phenols', 'Flavanoids', 'Nonflavanoid phenols', 'Proanthocyanins',
    'Color intensity', 'Hue', 'OD280/OD315 of diluted wines', 'Proline'
]
df = pd.read_csv('wine.data', header=None, names=column_names)

# Number of classes
num_classes = df['Class'].nunique()
print("Number of classes:", num_classes)

# Number of features (excluding the class label)
num_features = df.shape[1] - 1
print("Number of features:", num_features)

# Basic stats: min, max, mean, std
feature_stats = df.describe().T[['min', 'max', 'mean', 'std']]
print("\nFeature statistics:\n", feature_stats)

# Class distribution
class_counts = df['Class'].value_counts().sort_index()
print("\nClass distribution:\n", class_counts)



Number of classes: 3
Number of features: 13

Feature statistics:
                                  min      max        mean         std
Class                           1.00     3.00    1.938202    0.775035
Alcohol                        11.03    14.83   13.000618    0.811827
Malic acid                      0.74     5.80    2.336348    1.117146
Ash                             1.36     3.23    2.366517    0.274344
Alcalinity of ash              10.60    30.00   19.494944    3.339564
Magnesium                      70.00   162.00   99.741573   14.282484
Total phenols                   0.98     3.88    2.295112    0.625851
Flavanoids                      0.34     5.08    2.029270    0.998859
Nonflavanoid phenols            0.13     0.66    0.361854    0.124453
Proanthocyanins                 0.41     3.58    1.590899    0.572359
Color intensity                 1.28    13.00    5.058090    2.318286
Hue                             0.48     1.71    0.957449    0.228572
OD280/OD315 of diluted w

## Problem 1 - Part (a)
### Base Model Training and Evaluation


In [5]:
# Step 1: Drop the 'Class' column from the feature set and store it separately
# - Assign features to variable X
# - Subtract 1 from class labels to convert them to 0-based indexing
# - Assign class labels to variable y

X = df.drop('Class', axis=1).values
y = df['Class'].values - 1

In [6]:
# Step 2: Perform a train-test split (70% train, 30% test) using random_state=42

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

In [7]:
# Step 3: Use StandardScaler to normalize the features
# - Fit on X_train and transform both X_train and X_test

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [10]:
# Step 4: Use one-hot encoding for y_train and y_test
# - Use keras.utils.to_categorical
from tensorflow.keras.utils import to_categorical
y_train_categorical = to_categorical(y_train, num_classes=3)
y_test_categorical = to_categorical(y_test, num_classes=3)

In [11]:
# Step 5: Define a Sequential model with the following architecture:
# - Dense(64, relu)
# - Dense(32, relu)
# - Dense(3, softmax)  # 3-class classification

model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

In [12]:
# Step 6: Compile using Adam optimizer, categorical_crossentropy loss, and accuracy metric
# - Train for 20 epochs with batch_size=8 and validation_split=0.2

model.compile(
    optimizer=Adam(),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
history = model.fit(
    X_train, y_train_categorical,
    epochs=20,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

Epoch 1/20
13/13 [==============================] - 0s 6ms/step - loss: 1.1438 - accuracy: 0.4040 - val_loss: 0.9264 - val_accuracy: 0.5200
Epoch 2/20
13/13 [==============================] - 0s 1ms/step - loss: 0.8528 - accuracy: 0.5657 - val_loss: 0.7026 - val_accuracy: 0.7600
Epoch 3/20
13/13 [==============================] - 0s 1ms/step - loss: 0.6456 - accuracy: 0.8586 - val_loss: 0.5287 - val_accuracy: 1.0000
Epoch 4/20
13/13 [==============================] - 0s 1ms/step - loss: 0.4672 - accuracy: 0.9798 - val_loss: 0.3836 - val_accuracy: 1.0000
Epoch 5/20
13/13 [==============================] - 0s 1ms/step - loss: 0.3218 - accuracy: 0.9798 - val_loss: 0.2682 - val_accuracy: 0.9600
Epoch 6/20
13/13 [==============================] - 0s 1ms/step - loss: 0.2153 - accuracy: 0.9697 - val_loss: 0.1958 - val_accuracy: 0.9600
Epoch 7/20
13/13 [==============================] - 0s 1ms/step - loss: 0.1509 - accuracy: 0.9798 - val_loss: 0.1507 - val_accuracy: 0.9600
Epoch 8/20
13/13 [==

In [13]:
# Step 7: Evaluate the model on test data and print:
# - Accuracy
# - Classification report
# - Confusion matrix

test_loss, test_accuracy = model.evaluate(X_test, y_test_categorical, verbose=0)
print(f"\nTest accuracy: {test_accuracy:.4f}")

y_pred = np.argmax(model.predict(X_test), axis=1)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))



Test accuracy: 0.9815
2/2 [==============================] - 0s 827us/step

Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        18
           1       1.00      0.95      0.98        21
           2       1.00      1.00      1.00        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54


Confusion Matrix:
[[18  0  0]
 [ 1 20  0]
 [ 0  0 15]]


In [14]:
# Step 8: Convert the trained model to TFLite format and save it as "model_base.tflite"
# - Print the file size in kilobytes

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open("model_base.tflite", "wb") as f:
    f.write(tflite_model)

file_size_kb = os.path.getsize("model_base.tflite") / 1024
print(f"\nTFLite model size: {file_size_kb:.2f} KB")


INFO:tensorflow:Assets written to: /var/folders/8y/8hlxkkq54855jcpjr1pn6bfr0000gn/T/tmpmsgz33oe/assets


INFO:tensorflow:Assets written to: /var/folders/8y/8hlxkkq54855jcpjr1pn6bfr0000gn/T/tmpmsgz33oe/assets



TFLite model size: 14.09 KB


2025-07-18 20:14:52.586110: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2025-07-18 20:14:52.586127: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2025-07-18 20:14:52.586284: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/8y/8hlxkkq54855jcpjr1pn6bfr0000gn/T/tmpmsgz33oe
2025-07-18 20:14:52.586762: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2025-07-18 20:14:52.586766: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/8y/8hlxkkq54855jcpjr1pn6bfr0000gn/T/tmpmsgz33oe
2025-07-18 20:14:52.587628: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:382] MLIR V1 optimization pass is not enabled
2025-07-18 20:14:52.588111: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2025-07-18 20:14:52.607968: I tensorflow/cc/saved_model/loader.

## Problem 1 - Part (b)

### Quantization (int8, float16, dynamic range)


In [17]:
def quantize_and_evaluate(model, X_test, y_test_cat, quant_type, filename):
    # Create the TFLite converter from the trained Keras model
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # Set supported ops
    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS,
        tf.lite.OpsSet.SELECT_TF_OPS
    ]
    converter._experimental_lower_tensor_list_ops = False

    # Step 1: Apply quantization settings
    if quant_type == 'int8':
        # (a) Enable default optimizations
        # (b) Define a representative dataset generator (e.g., first 100 samples from X_train_scaled)
        # (c) Set inference_input_type and inference_output_type to tf.int8

        converter.optimizations = [tf.lite.Optimize.DEFAULT]

        def representative_data_gen():
            for i in range(min(100, X_test.shape[0])):
                yield [X_test[i].reshape(1, -1).astype(np.float32)]

        converter.representative_dataset = representative_data_gen
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8
        pass

    elif quant_type == 'float16':
        # (a) Enable default optimizations
        # (b) Set supported_types to [tf.float16]

        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]
        pass

    elif quant_type == 'dynamic':
        # (a) Enable default optimizations

        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        pass

    # Step 2: Convert the model and save it to the provided filename

    tflite_model = converter.convert()
    with open(filename, 'wb') as f:
        f.write(tflite_model)

    # Step 3: Run Inference
    # Complete the following:
    # - Use tf.lite.Interpreter to load the TFLite model
    # - Allocate tensors
    # - Get input/output tensor details
    # - If input is quantized (dtype=int8), quantize test input accordingly
    # - If output is quantized (dtype=int8), dequantize predictions
    # - Collect predictions into y_pred (use np.argmax to get class index)
    # - Compare with y_true = np.argmax(y_test_cat, axis=1)

    interpreter = tf.lite.Interpreter(model_path=filename)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    input_dtype = input_details[0]['dtype']
    output_dtype = output_details[0]['dtype']

    y_pred = []

    for i in range(len(X_test)):
        input_data = X_test[i].reshape(1, -1).astype(np.float32)

        if input_dtype == np.int8:
            scale, zero_point = input_details[0]['quantization']
            input_data = (input_data / scale + zero_point).astype(np.int8)

        interpreter.set_tensor(input_details[0]['index'], input_data)
        interpreter.invoke()
        output_data = interpreter.get_tensor(output_details[0]['index'])

        if output_dtype == np.int8:
            scale, zero_point = output_details[0]['quantization']
            output_data = (output_data.astype(np.float32) - zero_point) * scale

        y_pred.append(np.argmax(output_data))

    y_true = np.argmax(y_test_cat, axis=1)
    # Step 4: Report results
    print(f"\n📦 {quant_type.upper()} TFLite Model Size: {os.path.getsize(filename) / 1024:.2f} KB")

    # <-- Enter your code here: print classification_report and confusion_matrix <--#    print(f"✅ Classification Report ({quant_type}):")
    print(classification_report(y_true, y_pred))
    print("confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

In [18]:
# Step 5: Use the function above to create and evaluate three quantized models:
# - 'int8' → save as 'model_int8.tflite'
# - 'float16' → save as 'model_float16.tflite'
# - 'dynamic' → save as 'model_dynamic.tflite'

quantize_and_evaluate(model, X_test, y_test_categorical, 'int8', 'model_int8.tflite')
quantize_and_evaluate(model, X_test, y_test_categorical, 'float16', 'model_float16.tflite')
quantize_and_evaluate(model, X_test, y_test_categorical, 'dynamic', 'model_dynamic.tflite')

INFO:tensorflow:Assets written to: /var/folders/8y/8hlxkkq54855jcpjr1pn6bfr0000gn/T/tmpt97mhb4j/assets


INFO:tensorflow:Assets written to: /var/folders/8y/8hlxkkq54855jcpjr1pn6bfr0000gn/T/tmpt97mhb4j/assets



📦 INT8 TFLite Model Size: 5.75 KB
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        18
           1       1.00      0.95      0.98        21
           2       1.00      1.00      1.00        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

confusion Matrix:
[[18  0  0]
 [ 1 20  0]
 [ 0  0 15]]


/opt/anaconda3/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2025-07-18 20:19:25.611849: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2025-07-18 20:19:25.611860: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2025-07-18 20:19:25.611945: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/8y/8hlxkkq54855jcpjr1pn6bfr0000gn/T/tmpt97mhb4j
2025-07-18 20:19:25.612368: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2025-07-18 20:19:25.612371: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/8y/8hlxkkq54855jcpjr1pn6bfr0000gn/T/tmpt97mhb4j
2025-07-18 20:19:25.613553: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle

INFO:tensorflow:Assets written to: /var/folders/8y/8hlxkkq54855jcpjr1pn6bfr0000gn/T/tmp8koncmvx/assets


...

2025-07-18 20:19:25.865692: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2025-07-18 20:19:25.865706: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2025-07-18 20:19:25.865820: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/8y/8hlxkkq54855jcpjr1pn6bfr0000gn/T/tmp8koncmvx
2025-07-18 20:19:25.866302: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2025-07-18 20:19:25.866307: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/8y/8hlxkkq54855jcpjr1pn6bfr0000gn/T/tmp8koncmvx
2025-07-18 20:19:25.867789: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2025-07-18 20:19:25.888128: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/8y/8hlxkkq54855jcpjr1pn6bfr0000gn/T/tmp8koncmvx
2025-07-

INFO:tensorflow:Assets written to: /var/folders/8y/8hlxkkq54855jcpjr1pn6bfr0000gn/T/tmp5idzd4k4/assets


INFO:tensorflow:Assets written to: /var/folders/8y/8hlxkkq54855jcpjr1pn6bfr0000gn/T/tmp5idzd4k4/assets



📦 DYNAMIC TFLite Model Size: 8.20 KB
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        18
           1       1.00      0.95      0.98        21
           2       1.00      1.00      1.00        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

confusion Matrix:
[[18  0  0]
 [ 1 20  0]
 [ 0  0 15]]


2025-07-18 20:19:26.136295: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2025-07-18 20:19:26.136304: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2025-07-18 20:19:26.136389: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/8y/8hlxkkq54855jcpjr1pn6bfr0000gn/T/tmp5idzd4k4
2025-07-18 20:19:26.136805: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2025-07-18 20:19:26.136808: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/8y/8hlxkkq54855jcpjr1pn6bfr0000gn/T/tmp5idzd4k4
2025-07-18 20:19:26.138080: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2025-07-18 20:19:26.157083: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/8y/8hlxkkq54855jcpjr1pn6bfr0000gn/T/tmp5idzd4k4
2025-07-

## Problem 1 - Part (c)

### Pruning

In [20]:
# Step 1: Define a pruning schedule using tfmot.sparsity.keras.PolynomialDecay
# HINT:
# - Use initial_sparsity = 0.5 and final_sparsity = 0.7
# - Set end_step to total training steps (approx. dataset_size / batch_size * epochs)

batch_size = 8
epochs = 10
steps_per_epoch = int(np.ceil(X_train.shape[0] / batch_size))
end_step = steps_per_epoch * epochs

pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.5,
    final_sparsity=0.7,
    begin_step=0,
    end_step=end_step
)

In [21]:
# Step 2: Build a Sequential model with 3 pruned Dense layers:
# - Dense(64, relu)
# - Dense(32, relu)
# - Dense(3, softmax)
# Make sure each Dense layer is wrapped with prune_low_magnitude()

prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude

pruned_model = Sequential([
    prune_low_magnitude(Dense(64, activation='relu'), pruning_schedule=pruning_schedule),
    prune_low_magnitude(Dense(32, activation='relu'), pruning_schedule=pruning_schedule),
    prune_low_magnitude(Dense(3, activation='softmax'), pruning_schedule=pruning_schedule)
])
pruned_model.build(input_shape=(None, X_train.shape[1]))


In [22]:
# Step 3: Compile the model with categorical_crossentropy and accuracy
# - Train for 10 epochs with batch_size=8 and validation_split=0.2
# - Add tfmot.sparsity.keras.UpdatePruningStep() to the callbacks list

pruned_model.compile(
    optimizer=Adam(),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    tfmot.sparsity.keras.UpdatePruningStep()
]

history_pruned = pruned_model.fit(
    X_train, y_train_categorical,
    batch_size=batch_size,
    epochs=epochs,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1
)


Epoch 1/10
13/13 [==============================] - 1s 6ms/step - loss: 1.0065 - accuracy: 0.5960 - val_loss: 0.8383 - val_accuracy: 0.7200
Epoch 2/10
13/13 [==============================] - 0s 1ms/step - loss: 0.7270 - accuracy: 0.8182 - val_loss: 0.6356 - val_accuracy: 0.8800
Epoch 3/10
13/13 [==============================] - 0s 1ms/step - loss: 0.5235 - accuracy: 0.9192 - val_loss: 0.4657 - val_accuracy: 0.9200
Epoch 4/10
13/13 [==============================] - 0s 2ms/step - loss: 0.3633 - accuracy: 0.9798 - val_loss: 0.3453 - val_accuracy: 0.9200
Epoch 5/10
13/13 [==============================] - 0s 1ms/step - loss: 0.2513 - accuracy: 0.9798 - val_loss: 0.2683 - val_accuracy: 0.9200
Epoch 6/10
13/13 [==============================] - 0s 1ms/step - loss: 0.1743 - accuracy: 0.9899 - val_loss: 0.2264 - val_accuracy: 0.9200
Epoch 7/10
13/13 [==============================] - 0s 2ms/step - loss: 0.1282 - accuracy: 0.9899 - val_loss: 0.1895 - val_accuracy: 0.9200
Epoch 8/10
13/13 [==

In [23]:
# Step 4: Do any necessary post-processing (if needed) on the pruned model
# and save it using appropriate specifications to a TFLite file named "model_pruned.tflite".
# Print the final file size in KB.

final_pruned_model = tfmot.sparsity.keras.strip_pruning(pruned_model)

converter = tf.lite.TFLiteConverter.from_keras_model(final_pruned_model)
tflite_model = converter.convert()

with open("model_pruned.tflite", "wb") as f:
    f.write(tflite_model)

size_kb = os.path.getsize("model_pruned.tflite") / 1024
print(f"\n🧩 Pruned TFLite model size: {size_kb:.2f} KB")


INFO:tensorflow:Assets written to: /var/folders/8y/8hlxkkq54855jcpjr1pn6bfr0000gn/T/tmpzaykpeqa/assets


INFO:tensorflow:Assets written to: /var/folders/8y/8hlxkkq54855jcpjr1pn6bfr0000gn/T/tmpzaykpeqa/assets



🧩 Pruned TFLite model size: 14.12 KB


2025-07-18 20:22:12.099615: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2025-07-18 20:22:12.099626: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2025-07-18 20:22:12.099719: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/8y/8hlxkkq54855jcpjr1pn6bfr0000gn/T/tmpzaykpeqa
2025-07-18 20:22:12.100162: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2025-07-18 20:22:12.100166: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/8y/8hlxkkq54855jcpjr1pn6bfr0000gn/T/tmpzaykpeqa
2025-07-18 20:22:12.100998: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2025-07-18 20:22:12.109173: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/8y/8hlxkkq54855jcpjr1pn6bfr0000gn/T/tmpzaykpeqa
2025-07-

In [24]:
# Step 5: Evaluate using the stripped model
# - Use np.argmax for predictions
# - Print classification_report and confusion_matrix

y_pred_pruned = np.argmax(final_pruned_model.predict(X_test), axis=1)
y_true = np.argmax(y_test_categorical, axis=1)

print("\nClassification Report (Pruned Model):")
print(classification_report(y_true, y_pred_pruned))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred_pruned))


2/2 [==============================] - 0s 1ms/step

Classification Report (Pruned Model):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        18
           1       0.95      1.00      0.98        21
           2       1.00      0.93      0.97        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Confusion Matrix:
[[18  0  0]
 [ 0 21  0]
 [ 0  1 14]]


## Problem 1 - Part (d)

### Knowledge Distillation

In [25]:
# Step 1: Define a Sequential model for Student with:
# - Dense(32, relu)
# - Dense(16, relu)
# - Dense(3, softmax)

student_model = Sequential([
    Dense(32, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(16, activation='relu'),
    Dense(3, activation='softmax')
])


In [26]:
# Step 2: Use model.predict() on X_train_scaled to obtain teacher soft labels

teacher_preds_soft = model.predict(X_train)

4/4 [==============================] - 0s 628us/step


In [29]:
# Step 3:
# (a) Concatenate hard (y_train_cat) and soft (teacher_preds_soft) labels along axis=1
#     to create a combined label for distillation
# (b) Define a custom distillation_loss() function that:
#     - Splits y_true_combined into y_true_hard and y_true_soft
#     - Computes two losses (both using categorical_crossentropy)
#     - Combines them with a weight factor alpha = 0.5

# Hint: Use slicing [:, :3] and [:, 3:] to split the combined labels

y_train_combined = np.concatenate([y_train_categorical, teacher_preds_soft], axis=1)

def distillation_loss(y_true_combined, y_pred):

    # <-- Enter your code here: implement hard/soft label separation and weighted loss <--#
    y_true_hard = y_true_combined[:, :3]
    y_true_soft = y_true_combined[:, 3:]

    loss_fn = tf.keras.losses.CategoricalCrossentropy()
    loss_hard = loss_fn(y_true_hard, y_pred)
    loss_soft = loss_fn(y_true_soft, y_pred)

    return 0.5 * loss_hard + 0.5 * loss_soft
    pass

In [30]:
# Step 4: Compile the student model with Adam optimizer and distillation_loss
# - Train for 10 epochs, batch_size=8, validation_split=0.2

student_model.compile(
    optimizer=Adam(),
    loss=distillation_loss,
    metrics=['accuracy']
)

history_kd = student_model.fit(
    X_train, y_train_combined,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)


Epoch 1/10
13/13 [==============================] - 0s 6ms/step - loss: 1.0616 - accuracy: 0.5152 - val_loss: 1.0055 - val_accuracy: 0.6400
Epoch 2/10
13/13 [==============================] - 0s 1ms/step - loss: 0.9259 - accuracy: 0.7374 - val_loss: 0.8987 - val_accuracy: 0.8800
Epoch 3/10
13/13 [==============================] - 0s 1ms/step - loss: 0.8159 - accuracy: 0.8687 - val_loss: 0.8014 - val_accuracy: 0.9200
Epoch 4/10
13/13 [==============================] - 0s 1ms/step - loss: 0.7130 - accuracy: 0.8990 - val_loss: 0.7006 - val_accuracy: 0.9600
Epoch 5/10
13/13 [==============================] - 0s 1ms/step - loss: 0.6138 - accuracy: 0.9192 - val_loss: 0.5983 - val_accuracy: 0.9600
Epoch 6/10
13/13 [==============================] - 0s 1ms/step - loss: 0.5140 - accuracy: 0.9394 - val_loss: 0.4980 - val_accuracy: 0.9200
Epoch 7/10
13/13 [==============================] - 0s 1ms/step - loss: 0.4248 - accuracy: 0.9596 - val_loss: 0.4048 - val_accuracy: 0.9200
Epoch 8/10
13/13 [==

In [31]:
# Step 5: Convert the student model to TFLite
# - Use appropriate settings for classification models
# - Save as "model_kd.tflite"
# - Print the file size in KB

converter = tf.lite.TFLiteConverter.from_keras_model(student_model)
tflite_model_kd = converter.convert()

with open("model_kd.tflite", "wb") as f:
    f.write(tflite_model_kd)

size_kb = os.path.getsize("model_kd.tflite") / 1024
print(f"\n KD Student Model TFLite size: {size_kb:.2f} KB")

INFO:tensorflow:Assets written to: /var/folders/8y/8hlxkkq54855jcpjr1pn6bfr0000gn/T/tmpomiehlgs/assets


INFO:tensorflow:Assets written to: /var/folders/8y/8hlxkkq54855jcpjr1pn6bfr0000gn/T/tmpomiehlgs/assets



 KD Student Model TFLite size: 6.12 KB


2025-07-18 20:26:32.705480: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2025-07-18 20:26:32.705491: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2025-07-18 20:26:32.705584: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/8y/8hlxkkq54855jcpjr1pn6bfr0000gn/T/tmpomiehlgs
2025-07-18 20:26:32.706020: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2025-07-18 20:26:32.706025: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/8y/8hlxkkq54855jcpjr1pn6bfr0000gn/T/tmpomiehlgs
2025-07-18 20:26:32.707365: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2025-07-18 20:26:32.726662: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/8y/8hlxkkq54855jcpjr1pn6bfr0000gn/T/tmpomiehlgs
2025-07-

In [32]:
# Step 6: Use student_model.predict() to obtain predictions on X_test_scaled
# - Print classification_report and confusion_matrix

y_pred_kd = np.argmax(student_model.predict(X_test), axis=1)
y_true = np.argmax(y_test_categorical, axis=1)

print("\nClassification Report (Knowledge Distillation):")
print(classification_report(y_true, y_pred_kd))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred_kd))


2/2 [==============================] - 0s 938us/step

Classification Report (Knowledge Distillation):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        18
           1       1.00      0.90      0.95        21
           2       0.88      1.00      0.94        15

    accuracy                           0.96        54
   macro avg       0.96      0.97      0.96        54
weighted avg       0.97      0.96      0.96        54

Confusion Matrix:
[[18  0  0]
 [ 0 19  2]
 [ 0  0 15]]


## Problem 1 - Part (e)

### Possibility of Further Model Size Reduction

Can you **further reduce the model size** beyond the smallest model obtained in parts **(b)**, **(c)**, or **(d)**, **without sacrificing significant classification performance**?

Your task is to:

1. **Analyze and compare** the results from previous parts: Which model had the smallest size? Which performed best?

2. **Propose a strategy** that combines or enhances techniques learned so far.

3. **Implement** your proposed solution.

4. **Evaluate** the resulting model using both:
   - TFLite model size (in KB)
   - Classification performance (accuracy and report)

5. **Justify your results:**
   - If further size reduction is **not** possible without major loss of accuracy, explain why.
   - If you succeed in reducing the size **further**, highlight what change made the biggest difference.


### **Note:** If this part includes any code, please include it below. The related discussion should be submitted as part of your PDF that contains answers to all [Dis] questions in this assignment.


In [35]:
# <-- (if needed) Enter your code here <--#

# --- Train student again (if needed) ---
# Already done above with DistillationLoss

# --- INT8 Quantization on trained student model ---
def quantize_student_model(student_model, X_ref, filename='model_student_kd_int8.tflite'):
    converter = tf.lite.TFLiteConverter.from_keras_model(student_model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]

    def representative_data_gen():
        for i in range(min(100, X_ref.shape[0])):
            yield [X_ref[i].reshape(1, -1).astype(np.float32)]

    converter.representative_dataset = representative_data_gen
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8

    tflite_model = converter.convert()
    with open(filename, 'wb') as f:
        f.write(tflite_model)

    return filename

# Convert
quantized_student_file = quantize_student_model(student_model, X_test)

def evaluate_tflite_model(filepath, X_test, y_test_cat):
    interpreter = tf.lite.Interpreter(model_path=filepath)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    input_dtype = input_details[0]['dtype']
    output_dtype = output_details[0]['dtype']

    y_pred = []

    for i in range(len(X_test)):
        input_data = X_test[i].reshape(1, -1).astype(np.float32)

        if input_dtype == np.int8:
            scale, zero_point = input_details[0]['quantization']
            input_data = (input_data / scale + zero_point).astype(np.int8)

        interpreter.set_tensor(input_details[0]['index'], input_data)
        interpreter.invoke()
        output_data = interpreter.get_tensor(output_details[0]['index'])

        if output_dtype == np.int8:
            scale, zero_point = output_details[0]['quantization']
            output_data = (output_data.astype(np.float32) - zero_point) * scale

        y_pred.append(np.argmax(output_data))

    y_true = np.argmax(y_test_cat, axis=1)

    print(f"\n📦 TFLite Model Size: {os.path.getsize(filepath) / 1024:.2f} KB")
    print("\n📊 Classification Report:")
    print(classification_report(y_true, y_pred))
    print("🧩 Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

# Evaluate
evaluate_tflite_model(quantized_student_file, X_test, y_test_categorical)



INFO:tensorflow:Assets written to: /var/folders/8y/8hlxkkq54855jcpjr1pn6bfr0000gn/T/tmptlvd81kq/assets


INFO:tensorflow:Assets written to: /var/folders/8y/8hlxkkq54855jcpjr1pn6bfr0000gn/T/tmptlvd81kq/assets



📦 TFLite Model Size: 3.66 KB

📊 Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        18
           1       1.00      0.90      0.95        21
           2       0.88      1.00      0.94        15

    accuracy                           0.96        54
   macro avg       0.96      0.97      0.96        54
weighted avg       0.97      0.96      0.96        54

🧩 Confusion Matrix:
[[18  0  0]
 [ 0 19  2]
 [ 0  0 15]]


/opt/anaconda3/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2025-07-18 20:35:01.137215: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2025-07-18 20:35:01.137225: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2025-07-18 20:35:01.137310: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/8y/8hlxkkq54855jcpjr1pn6bfr0000gn/T/tmptlvd81kq
2025-07-18 20:35:01.137739: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2025-07-18 20:35:01.137743: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/8y/8hlxkkq54855jcpjr1pn6bfr0000gn/T/tmptlvd81kq
2025-07-18 20:35:01.138905: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle

# Problem 2: : Exploring Edge Impulse (40 points)

### **Note:** This problem consists entirely of [Dis] questions. Please provide your answers in the same PDF file that contains responses to the other [Dis] questions in this assignment.
